# p-IgGen: Does the Extraction Convention Explain the §1r-D Contradiction?

## The contradiction

Two of our own runs disagree on the same model at the same dose:

| run | α_rel | length | layer 1 collapse |
|---|---|---|---|
| `41` | 0.400 | 49.0 residues (effectively fixed) | **0 / 50** |
| `57` | 0.400 | 50 residues (fixed by construction) | **17 / 100** |

If the true rate were 0.17, P(0 of 50) ≈ 1e-4. **This is not sampling noise**, and it is not the
§1p length artifact — `41`'s sequences were already all exactly 49.0 residues, verified.

`57`'s result is the basis of §1r-C, the project's cleanest early-vs-late layer finding. It cannot
be reported while an earlier run of ours contradicts it.

## The prime suspect

`41` built its steering vector from `out.hidden_states[L]`; `57` used a forward hook on block `L`.
§1q-C proved these give **cosine +0.9985** and statistically identical steering — **but that was
measured on ProtGPT2, which has 36 layers, where one block is 2.8% of the network.**

**p-IgGen has FOUR layers. One block is 25%.** The cosine has never been measured here, and there
is no reason to expect the ProtGPT2 number to transfer.

## What this notebook does

**Part 1 — the indexing proof (no folding, seconds).** Confirm on *this* model that a hook on block
L returns exactly `hidden_states[L+1]`, as it did on ProtGPT2.

**Part 2 — the two cosines (no folding beyond the pool).**
- **cosine(v_hook, v_legacy)** at layers 1 and 3. If it is near 1, the convention is not the
  explanation and the discrepancy must lie in the vector itself. If it is low, the two runs were
  applying materially different interventions and §1r-D is solved.
- **Vector stability:** build vectors from two disjoint halves of the utility-matched pool and take
  their cosine, repeated over several random splits. **This measures how reproducible the steering
  vector is at all** — the other candidate explanation, since `41` used a 200-candidate pool and
  `57` used 150. A project that scales vectors to a fixed α_rel has never checked whether the
  *direction* is stable, and it matters for every steering result here.

**Part 3 — the decisive steering test.** Both conventions, both layers, N=100 per arm, fixed 50
residues: CONTROL, L1_HOOK, L3_HOOK, L1_LEGACY, L3_LEGACY.

This answers two things at once:
1. Does the legacy vector reproduce `41`'s near-zero collapse while the hook vector reproduces
   `57`'s 17%? That would settle §1r-D outright.
2. **Does p-IgGen's layer difference survive the convention choice?** §1r-C is only safe to report
   if it holds under both.

## Pre-registered readings

| outcome | reading |
|---|---|
| low cosine **and** L1_LEGACY ≈ 0% while L1_HOOK ≈ 17% | §1r-D solved: convention. `41` and `57` measured different interventions. Report §1r-C under the stated convention and say so. |
| high cosine **and** both conventions ≈ 17% | Convention is not it. `41`'s 0/50 becomes the anomaly — suspect its pool/vector draw. Vector stability in Part 2 then carries the explanation. |
| layer difference present under **both** conventions | §1r-C is robust; report it. |
| layer difference present under **only one** | §1r-C is convention-dependent — a serious caveat that must be stated in the paper. |
| vector stability cosine well below 1 | **A finding in its own right**, affecting every steering result in this project: the contrastive direction is not reproducible across pools, and α_rel matching controls only magnitude. |

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect **~1.5–2 hours**
(150 pool folds + 500 condition folds, all at ≤50 residues, so folding is fast).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.stats import fisher_exact, mannwhitneyu

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ProtGPT2's own natural v_L norm from notebook 03. Used ONLY to define the anchor and to report
# how far the old absolute-norm runs were from matched -- never to scale anything in this notebook.
REFERENCE_NORM = 583.998

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())
print("Model under test: opig/p-IgGen")


Setup complete. CUDA available: True
Model under test: opig/p-IgGen


In [2]:
# --- The same common probe set as 34/38/40/41/47: real UniProt fragments. Using one shared probe
#     set across every repair notebook is what makes their alpha_rel values directly comparable
#     rather than each notebook being its own island. ---

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!! Falling back to hardcoded reference sequences so the run can proceed, but the probe")
    print("!! set will then differ from 34/38/40/41/47 -- say so if you log this result.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        L = min(frag_len, len(seq))
        start = rng.randint(0, max(1, len(seq) - L + 1))
        probes.append(seq[start:start + L])
    return probes

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

probe_seqs = build_probe_set(reference_seqs, n_probes=40)
print(f"\nBuilt {len(probe_seqs)} common probe fragments from {len(reference_seqs)} source proteins.")

def get_layer_module(model, path, idx):
    obj = model
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj[idx]

def measure_resid_norm_mod(model, layer_module, tokenize_fn, seqs):
    # Mean per-position residual-stream norm at one layer. This is the ‖h‖ that alpha_rel
    # divides by, and measuring it per-layer is the entire point of this notebook.
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            enc = tokenize_fn(seq)
            captured.clear()
            with torch.no_grad():
                model(**enc)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

print("Probe set and residual-norm measurement ready.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments from 12 source proteins.
Probe set and residual-norm measurement ready.


In [3]:
# --- The anchor: ProtGPT2's own layer-12 alpha_rel, the SAME anchor 38/40/41/47 used. Every dose
#     in this notebook is a multiple of this number, so the result sits on the same scale as the
#     rest of the repaired cross-model picture instead of being internally-consistent-only. ---

print(f"Loading ProtGPT2 on {device} to measure the anchor...")
anchor_tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
anchor_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
anchor_model.eval()

def anchor_tok(s):
    return anchor_tokenizer(s, return_tensors="pt", truncation=True, max_length=256).to(device)

h_protgpt2_l12 = measure_resid_norm_mod(
    anchor_model, get_layer_module(anchor_model, "transformer.h", 12), anchor_tok, probe_seqs)
anchor_alpha_rel = REFERENCE_NORM / h_protgpt2_l12

print(f"ProtGPT2 layer 12 mean residual-stream norm (this run's probes): {h_protgpt2_l12:.2f}")
print(f"anchor_alpha_rel = {anchor_alpha_rel:.4f}")
print("(34 measured 2921.53 -> 0.2000; 38/40/41/47 each reproduced ~0.2000. A close value here")
print(" confirms the anchor is a stable measured quantity rather than a fluke of one run.)")

del anchor_model, anchor_tokenizer
clear_gpu()
print("\nProtGPT2 anchor freed from GPU.")


Loading ProtGPT2 on cuda to measure the anchor...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtGPT2 layer 12 mean residual-stream norm (this run's probes): 2919.92
anchor_alpha_rel = 0.2000
(34 measured 2921.53 -> 0.2000; 38/40/41/47 each reproduced ~0.2000. A close value here
 confirms the anchor is a stable measured quantity rather than a fluke of one run.)

ProtGPT2 anchor freed from GPU.


In [4]:
# --- Scoring + ESMFold. Identical to 24/26/27/41, including the length-aware OOM retry added in
#     39/40 and the fold_ok tracking added after the §1l collapse-metric hole. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and length-aware ESMFold evaluator ready.")


Scoring functions and length-aware ESMFold evaluator ready.


In [5]:
# --- p-IgGen loading and generation, matching 27/41/57. ---

HOOK_PATH = "gpt_neox.layers"
LAYER_EARLY, LAYER_LATE = 1, 3
TARGET_LEN = 50
N_PER_CONDITION = 100
DOSE = 2.0          # 2x the anchor = alpha_rel 0.400, the dose 41 and 57 disagree about

print(f"Loading p-IgGen on {device}...")
tokenizer = AutoTokenizer.from_pretrained("opig/p-IgGen")
plm_model = AutoModelForCausalLM.from_pretrained("opig/p-IgGen").to(device)
plm_model.eval()

N_LAYERS = plm_model.config.num_hidden_layers
PROMPT_CHAR = "1"
EOS_ID = tokenizer.encode("2")[0]
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else EOS_ID
print(f"p-IgGen: {N_LAYERS} layers (one block = {100.0/N_LAYERS:.0f}% of the network).")
print(f"Compare against ProtGPT2's 36 layers, where one block is 2.8% -- this is the whole reason")
print(f"1q-C's 'the off-by-one is harmless' may not transfer to this model.")
print(f"eos id for '2' = {EOS_ID}, pad = {PAD_ID}")

def model_tok(s):
    return tokenizer(s, return_tensors="pt", truncation=True, max_length=256).to(device)

def clean_aa(text):
    return "".join(a for a in text.replace(" ", "") if a in VALID_AA)

def generate_natural(n, seed):
    torch.manual_seed(seed)
    recs = []
    for _ in range(n):
        inputs = tokenizer(PROMPT_CHAR, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = plm_model.generate(**inputs, max_length=51, do_sample=True,
                                         temperature=1.2, eos_token_id=EOS_ID,
                                         pad_token_id=PAD_ID)
        raw = clean_aa(tokenizer.decode(out_ids[0], skip_special_tokens=False))
        recs.append({"prompt": PROMPT_CHAR, "sequence": raw, "clean": raw,
                     "entropy": calculate_entropy(raw)})
    clear_gpu()
    return recs

N_CANDIDATES = 150
print(f"\n=== Generating N={N_CANDIDATES} natural candidates ===")
candidate_records = generate_natural(N_CANDIDATES, seed=505)
lens = [len(r["clean"]) for r in candidate_records]
print(f"natural usable length: mean {np.mean(lens):.1f}  (41 recorded exactly 49.0)")
print(f"example: {candidate_records[0]['clean'][:70]}")


Loading p-IgGen on cuda...


config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

p-IgGen: 4 layers (one block = 25% of the network).
Compare against ProtGPT2's 36 layers, where one block is 2.8% -- this is the whole reason
1q-C's 'the off-by-one is harmless' may not transfer to this model.
eos id for '2' = 24, pad = 0

=== Generating N=150 natural candidates ===
natural usable length: mean 50.0  (41 recorded exactly 49.0)
example: QLQLQESGPGLVKPSDTLSLTCSVSGGSMNTQTHYWGWIRQPPGKSLEWI


In [6]:
# =============================================================================================
# PART 1 -- the indexing proof, on THIS model. No folding.
# =============================================================================================

for L in (LAYER_EARLY, LAYER_LATE):
    cap = {}
    lm = get_layer_module(plm_model, HOOK_PATH, L)
    h = lm.register_forward_hook(
        lambda m, i, o: cap.__setitem__("h", (o[0] if isinstance(o, (tuple, list)) else o).detach()))
    with torch.no_grad():
        out = plm_model(**model_tok(candidate_records[0]["clean"]), output_hidden_states=True)
    h.remove()
    d_same = (cap["h"] - out.hidden_states[L]).abs().max().item()
    d_next = (cap["h"] - out.hidden_states[L + 1]).abs().max().item()
    print(f"layer {L}: hook vs hidden_states[{L}] max|diff| = {d_same:.6f}")
    print(f"         hook vs hidden_states[{L+1}] max|diff| = {d_next:.6f}")
print(f"len(hidden_states) = {len(out.hidden_states)}  (n_layers + 1 = {N_LAYERS + 1})")
print()
print("Same relationship as ProtGPT2: a hook on block L == hidden_states[L+1]. So 41 built its")
print("vector one block earlier than it injected it, on a model where one block is a QUARTER of")
print("the network.")
del out, cap
clear_gpu()


layer 1: hook vs hidden_states[1] max|diff| = 1.881555
         hook vs hidden_states[2] max|diff| = 0.000000
layer 3: hook vs hidden_states[3] max|diff| = 8.967970
         hook vs hidden_states[4] max|diff| = 5.848316
len(hidden_states) = 5  (n_layers + 1 = 5)

Same relationship as ProtGPT2: a hook on block L == hidden_states[L+1]. So 41 built its
vector one block earlier than it injected it, on a model where one block is a QUARTER of
the network.


In [7]:
# --- Fold the pool so the contrastive sets can be utility-matched exactly as 27/41/57 did. ---

del plm_model
clear_gpu()

evaluator = StructuralEvaluatorPTM()
print("Folding the candidate pool...")
candidate_records = fold_records_ptm(candidate_records, evaluator)
del evaluator
clear_gpu()

valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
pool_collapse = float(np.mean([r["collapse"] for r in valid_candidates]))
pool_plddt = float(np.mean([r["plddt"] for r in valid_candidates]))
print(f"\n{len(valid_candidates)}/{len(candidate_records)} folded. "
      f"collapse {pool_collapse:.1%}, mean pLDDT {pool_plddt:.2f}")
print("(1i recorded 0.0% / 74.36; 57's pool 0.0% / 73.90 -- close values confirm comparability.)")


Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding the candidate pool...

150/150 folded. collapse 0.0%, mean pLDDT 74.29
(1i recorded 0.0% / 74.36; 57's pool 0.0% / 73.90 -- close values confirm comparability.)


In [8]:
# --- Utility-matching, unchanged from 24/26/27/41/47. Trims the extreme-repetition tails until
#     D+ and D- have comparable structural utility, so the resulting difference vector encodes
#     repetition rather than "good protein vs. junk". ---

QUANTILE = 0.30
UTILITY_TOLERANCE = 0.05

sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(sorted_by_rep) * QUANTILE))
d_minus_raw = sorted_by_rep[:n_side]
d_plus_raw = sorted_by_rep[-n_side:]

def utility_match(pool_a, pool_b, tolerance, max_iters=200):
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r["utility_score"] for r in a])
        mean_b = np.mean([r["utility_score"] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r["utility_score"]); a.pop(0)
        else:
            b.sort(key=lambda r: r["utility_score"]); b.pop(0)
    return a, b

d_plus, d_minus = utility_match(d_plus_raw, d_minus_raw, UTILITY_TOLERANCE)
util_gap = abs(np.mean([r["utility_score"] for r in d_plus]) - np.mean([r["utility_score"] for r in d_minus]))
rep_gap = abs(np.mean([r["repetition_score"] for r in d_plus]) - np.mean([r["repetition_score"] for r in d_minus]))
print(f"After utility-matching: D+ n={len(d_plus)}, D- n={len(d_minus)}")
print(f"  utility gap {util_gap:.3f} (target <= {UTILITY_TOLERANCE})")
print(f"  repetition separation {rep_gap:.3f} (this is the signal the vector should encode)")


After utility-matching: D+ n=45, D- n=45
  utility gap 0.019 (target <= 0.05)
  repetition separation 0.035 (this is the signal the vector should encode)


In [9]:
# =============================================================================================
# PART 2 -- the two cosines.
#   (a) convention:  cosine(v_hook, v_legacy)
#   (b) stability:   cosine(v from half A, v from half B) over repeated random splits
# =============================================================================================

print(f"Reloading p-IgGen...")
plm_model = AutoModelForCausalLM.from_pretrained("opig/p-IgGen").to(device)
plm_model.eval()

def act_hook(seqs, layer):
    cap, acts = {}, []
    lm = get_layer_module(plm_model, HOOK_PATH, layer)
    handle = lm.register_forward_hook(
        lambda m, i, o: cap.__setitem__("h", (o[0] if isinstance(o, (tuple, list)) else o).detach()))
    try:
        for s in seqs:
            cap.clear()
            with torch.no_grad():
                plm_model(**model_tok(s))
            if "h" in cap:
                acts.append(cap["h"].float().mean(dim=1).squeeze(0).cpu())
    finally:
        handle.remove()
    return torch.stack(acts)

def act_legacy(seqs, layer):
    acts = []
    for s in seqs:
        with torch.no_grad():
            out = plm_model(**model_tok(s), output_hidden_states=True)
        acts.append(out.hidden_states[layer].float().mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

def cosine(a, b):
    return float(torch.dot(a, b) / (a.norm() * b.norm()))

dp = [r["sequence"] for r in d_plus]
dm = [r["sequence"] for r in d_minus]

# Cache activations once per (convention, layer); everything below is arithmetic on them.
A = {}
for L in (LAYER_EARLY, LAYER_LATE):
    A[("hook", L, "+")] = act_hook(dp, L)
    A[("hook", L, "-")] = act_hook(dm, L)
    A[("legacy", L, "+")] = act_legacy(dp, L)
    A[("legacy", L, "-")] = act_legacy(dm, L)
print("activations cached")

BAR = "=" * 96
print()
print(BAR)
print("2a -- DOES THE EXTRACTION CONVENTION CHANGE THE DIRECTION?")
print(BAR)
vectors, conv_cos = {}, {}
for L in (LAYER_EARLY, LAYER_LATE):
    v_hook = A[("hook", L, "+")].mean(0) - A[("hook", L, "-")].mean(0)
    v_leg = A[("legacy", L, "+")].mean(0) - A[("legacy", L, "-")].mean(0)
    vectors[("hook", L)] = v_hook
    vectors[("legacy", L)] = v_leg
    conv_cos[L] = cosine(v_hook, v_leg)
    print(f"layer {L}:  ||v_hook|| = {v_hook.norm():.4f}   ||v_legacy|| = {v_leg.norm():.4f}")
    print(f"          cosine(v_hook, v_legacy) = {conv_cos[L]:+.4f}")
print()
print(f"ProtGPT2's value at layer 12 was +0.9985 (1q-C), on a 36-layer model.")
worst = min(conv_cos.values())
if worst > 0.95:
    print("  ==> The conventions agree here too. The off-by-one is NOT the explanation for 1r-D,")
    print("      and the discrepancy must lie in the vector itself -- see 2b.")
elif worst > 0.70:
    print("  ==> Partial agreement. The conventions are not interchangeable on this model;")
    print("      Part 3 decides whether the difference is large enough to matter in practice.")
else:
    print("  ==> MATERIALLY DIFFERENT DIRECTIONS. 41 and 57 were applying different")
    print("      interventions, which is a sufficient explanation for 1r-D. It also means")
    print("      1q-C's 'harmless' finding is ProtGPT2-specific and must be scoped in the paper.")

print()
print(BAR)
print("2b -- IS THE STEERING VECTOR REPRODUCIBLE AT ALL?")
print(BAR)
print("Split the utility-matched pool into two disjoint halves, build a vector from each, and")
print("take their cosine. This has never been measured in this project, and it bounds how much")
print("of any cross-run disagreement is just vector noise.")
print()
rng = np.random.RandomState(2026)
N_SPLITS = 20
stab = {}
for conv in ("hook", "legacy"):
    for L in (LAYER_EARLY, LAYER_LATE):
        pos, neg = A[(conv, L, "+")], A[(conv, L, "-")]
        cos_list = []
        for _ in range(N_SPLITS):
            ip = rng.permutation(len(pos))
            inn = rng.permutation(len(neg))
            pa, pb = ip[: len(pos) // 2], ip[len(pos) // 2:]
            na, nb = inn[: len(neg) // 2], inn[len(neg) // 2:]
            va = pos[pa].mean(0) - neg[na].mean(0)
            vb = pos[pb].mean(0) - neg[nb].mean(0)
            cos_list.append(cosine(va, vb))
        stab[(conv, L)] = (float(np.mean(cos_list)), float(np.std(cos_list)),
                           float(np.min(cos_list)))
        m, sd, mn = stab[(conv, L)]
        print(f"  {conv:7s} layer {L}:  split-half cosine {m:+.4f} +- {sd:.4f}   (min {mn:+.4f})")
print()
min_stab = min(v[0] for v in stab.values())
if min_stab > 0.90:
    print("  ==> The vector is highly reproducible across pool draws. Pool composition cannot")
    print("      explain 1r-D either -- which would leave the convention (2a) as the only")
    print("      remaining candidate.")
elif min_stab > 0.60:
    print("  ==> Moderately reproducible. A different pool gives a noticeably different direction,")
    print("      so pool draw is a live explanation for 41-vs-57 alongside the convention.")
else:
    print("  ==> THE VECTOR IS NOT REPRODUCIBLE. Two halves of the SAME pool give substantially")
    print("      different directions. This is a finding in its own right and affects every")
    print("      steering result in the project: alpha_rel matching controls magnitude, and")
    print("      nothing has ever controlled direction. Report it.")


Reloading p-IgGen...


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

activations cached

2a -- DOES THE EXTRACTION CONVENTION CHANGE THE DIRECTION?
layer 1:  ||v_hook|| = 0.5918   ||v_legacy|| = 0.3438
          cosine(v_hook, v_legacy) = +0.1564
layer 3:  ||v_hook|| = 4.4147   ||v_legacy|| = 1.1701
          cosine(v_hook, v_legacy) = +0.2209

ProtGPT2's value at layer 12 was +0.9985 (1q-C), on a 36-layer model.
  ==> MATERIALLY DIFFERENT DIRECTIONS. 41 and 57 were applying different
      interventions, which is a sufficient explanation for 1r-D. It also means
      1q-C's 'harmless' finding is ProtGPT2-specific and must be scoped in the paper.

2b -- IS THE STEERING VECTOR REPRODUCIBLE AT ALL?
Split the utility-matched pool into two disjoint halves, build a vector from each, and
take their cosine. This has never been measured in this project, and it bounds how much
of any cross-run disagreement is just vector noise.

  hook    layer 1:  split-half cosine +0.7376 +- 0.1037   (min +0.4982)
  hook    layer 3:  split-half cosine +0.8389 +- 0.0905   (min 

In [10]:
# =============================================================================================
# PART 3 -- the decisive steering test. Both conventions, both layers, N=100, fixed length.
# =============================================================================================

h_model, matched_norms = {}, {}
for L in (LAYER_EARLY, LAYER_LATE):
    lm = get_layer_module(plm_model, HOOK_PATH, L)
    h_model[L] = measure_resid_norm_mod(plm_model, lm, model_tok, probe_seqs)
    matched_norms[L] = anchor_alpha_rel * h_model[L]
    print(f"layer {L}: ||h|| = {h_model[L]:.3f} (34: {6.14 if L == 1 else 21.81}), "
          f"matched norm {matched_norms[L]:.4f}, alpha_rel {matched_norms[L]/h_model[L]:.6f}")
    assert abs(matched_norms[L] / h_model[L] - anchor_alpha_rel) < 1e-6, "alpha_rel wrong"

scaled = {}
for (conv, L), v in vectors.items():
    scaled[(conv, L)] = (v * (matched_norms[L] / v.norm().item())).to(device)

MIN_NEW, MAX_NEW = 200, 280
try:
    _t = tokenizer(PROMPT_CHAR, return_tensors="pt").to(device)
    with torch.no_grad():
        plm_model.generate(**_t, min_new_tokens=5, max_new_tokens=8, do_sample=True,
                           temperature=1.2, pad_token_id=PAD_ID, eos_token_id=EOS_ID)
    SUPPORTS_MIN_NEW = True
except TypeError as e:
    SUPPORTS_MIN_NEW = False
    print(f"!! min_new_tokens unsupported ({e})")
print(f"min_new_tokens supported: {SUPPORTS_MIN_NEW}")

def generate_fixed(n, layer, vec, seed, max_attempts=3):
    torch.manual_seed(seed)
    v = None if vec is None else vec.to(device)
    def hook(module, inp, out):
        if v is None:
            return out
        if isinstance(out, tuple):
            return (out[0] + v,) + out[1:]
        return out + v
    lm = get_layer_module(plm_model, HOOK_PATH, layer)
    recs, n_short = [], 0
    for _ in range(n):
        best = ""
        for attempt in range(max_attempts):
            handle = lm.register_forward_hook(hook)
            inputs = tokenizer(PROMPT_CHAR, return_tensors="pt").to(device)
            kw = dict(do_sample=True, temperature=1.2, pad_token_id=PAD_ID, eos_token_id=EOS_ID,
                      max_new_tokens=MAX_NEW + attempt * 120)
            if SUPPORTS_MIN_NEW:
                kw["min_new_tokens"] = MIN_NEW + attempt * 120
            with torch.no_grad():
                out_ids = plm_model.generate(**inputs, **kw)
            handle.remove()
            cl = clean_aa(tokenizer.decode(out_ids[0], skip_special_tokens=False))
            if len(cl) > len(best):
                best = cl
            if len(best) >= TARGET_LEN:
                break
        if len(best) < TARGET_LEN:
            n_short += 1
        seq = best[:TARGET_LEN]
        recs.append({"sequence": seq, "entropy": calculate_entropy(seq),
                     "reached_target": len(best) >= TARGET_LEN, "raw_available": len(best)})
    clear_gpu()
    if n_short:
        print(f"    !! {n_short}/{n} short of {TARGET_LEN}; kept and flagged.")
    return recs

conditions, cond_meta = {}, {}
print(f"\n=== CONTROL ===")
conditions["CONTROL"] = generate_fixed(N_PER_CONDITION, LAYER_EARLY, None, 8000)
cond_meta["CONTROL"] = {"layer": None, "convention": "none"}
seed = 8000
for conv in ("hook", "legacy"):
    for L in (LAYER_EARLY, LAYER_LATE):
        seed += 137
        name = f"L{L}_{conv.upper()}"
        print(f"=== {name} (alpha_rel {DOSE * anchor_alpha_rel:.3f}) ===")
        conditions[name] = generate_fixed(N_PER_CONDITION, L, scaled[(conv, L)] * DOSE, seed)
        cond_meta[name] = {"layer": L, "convention": conv}

del plm_model
clear_gpu()

evaluator2 = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    conditions[name] = fold_records_ptm(conditions[name], evaluator2)
del evaluator2
clear_gpu()

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    hh = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, c - hh), min(1.0, c + hh))

summary = {}
print()
print("{:14s}{:>6}{:>9}{:>9}{:>9}{:>22}".format(
    "Condition", "N", "FoldOK", "Entropy", "pLDDT", "Collapse% [95% CI]"))
print("-" * 70)
for name, recs in conditions.items():
    n = len(recs)
    k = int(np.sum([r["collapse"] for r in recs]))
    ps = [r["plddt"] for r in recs if r["plddt"] > 0]
    lo, hi = wilson_ci(k, n)
    summary[name] = {"k": k, "n": n, "rate": k / n, "ci": (lo, hi),
                     "fold_ok": sum(1 for r in recs if r["fold_ok"]),
                     "plddt_mean": float(np.mean(ps)) if ps else 0.0, "plddts": ps,
                     "entropy": float(np.mean([r["entropy"] for r in recs])),
                     "mean_len": float(np.mean([len(r["sequence"]) for r in recs]))}
    s = summary[name]
    print("{:14s}{:6d}{:>9}{:9.3f}{:9.2f}{:>13.1f}% [{:.0%},{:.0%}]".format(
        name, n, f"{s['fold_ok']}/{n}", s["entropy"], s["plddt_mean"], s["rate"] * 100, lo, hi))


layer 1: ||h|| = 6.128 (34: 6.14), matched norm 1.2257, alpha_rel 0.200005
layer 3: ||h|| = 21.406 (34: 21.81), matched norm 4.2812, alpha_rel 0.200005
min_new_tokens supported: True

=== CONTROL ===
=== L1_HOOK (alpha_rel 0.400) ===
=== L3_HOOK (alpha_rel 0.400) ===
=== L1_LEGACY (alpha_rel 0.400) ===
=== L3_LEGACY (alpha_rel 0.400) ===
Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding L1_HOOK...
Folding L3_HOOK...
Folding L1_LEGACY...
Folding L3_LEGACY...

Condition          N   FoldOK  Entropy    pLDDT    Collapse% [95% CI]
----------------------------------------------------------------------
CONTROL          100  100/100    3.767    73.81          3.0% [1%,8%]
L1_HOOK          100  100/100    3.729    71.33          9.0% [5%,16%]
L3_HOOK          100  100/100    3.781    73.01          4.0% [2%,10%]
L1_LEGACY        100  100/100    3.748    70.55          9.0% [5%,16%]
L3_LEGACY        100  100/100    3.775    74.19          1.0% [0%,5%]


In [11]:
# --- The verdicts. ---

def compare(a, b):
    _, p_c = fisher_exact([[a["k"], a["n"] - a["k"]], [b["k"], b["n"] - b["k"]]])
    if len(a["plddts"]) >= 3 and len(b["plddts"]) >= 3:
        _, p_p = mannwhitneyu(a["plddts"], b["plddts"], alternative="two-sided")
    else:
        p_p = float("nan")
    return p_c, p_p, a["rate"] - b["rate"], a["plddt_mean"] - b["plddt_mean"]

def holm2(pa, pb, alpha=0.05):
    order = sorted([(0, pa), (1, pb)],
                   key=lambda t: (float("inf") if math.isnan(t[1]) else t[1]))
    out, still = [False, False], True
    for rank, (idx, p) in enumerate(order):
        ok = still and (not math.isnan(p)) and p <= alpha / (2 - rank)
        out[idx] = ok
        if not ok:
            still = False
    return out[0], out[1]

MIN_PLDDT_SHIFT, MIN_RATE_SHIFT = 3.0, 0.10
ctrl = summary["CONTROL"]
BAR = "=" * 96

print(BAR)
print("VERDICT 1 -- DOES THE CONVENTION REPRODUCE THE 41-vs-57 SPLIT?")
print(BAR)
print("41 saw 0/50 collapse at layer 1 (alpha_rel 0.400); 57 saw 17/100 with the HOOK vector.")
print()
conv_rows = []
for L in (LAYER_EARLY, LAYER_LATE):
    hk, lg = summary[f"L{L}_HOOK"], summary[f"L{L}_LEGACY"]
    p_c, p_p, d_r, d_p = compare(hk, lg)
    differs = ((p_c < 0.05 and abs(d_r) >= MIN_RATE_SHIFT)
               or ((not math.isnan(p_p)) and p_p < 0.05 and abs(d_p) >= MIN_PLDDT_SHIFT))
    conv_rows.append({"layer": L, "hook_rate": hk["rate"], "legacy_rate": lg["rate"],
                      "hook_plddt": hk["plddt_mean"], "legacy_plddt": lg["plddt_mean"],
                      "diff_rate": d_r, "diff_plddt": d_p, "p_collapse": p_c,
                      "p_plddt": p_p, "conventions_differ": bool(differs)})
    print(f"layer {L}:  hook {hk['rate']:.0%} / {hk['plddt_mean']:.2f}    "
          f"legacy {lg['rate']:.0%} / {lg['plddt_mean']:.2f}")
    print(f"          difference {d_r:+.1%} collapse (p={p_c:.4g}), "
          f"{d_p:+.2f} pLDDT (p={p_p:.4g})   -> {'DIFFERS' if differs else 'same'}")
conv_matters = any(r["conventions_differ"] for r in conv_rows)
print()
if conv_matters:
    print("  ==> THE CONVENTION MATTERS on p-IgGen. 41 and 57 were applying materially different")
    print("      interventions, which resolves 1r-D. State the convention explicitly wherever a")
    print("      p-IgGen steering number is reported, and scope 1q-C to ProtGPT2.")
else:
    print("  ==> The convention does NOT change the outcome here either. 1r-D is therefore NOT")
    print("      explained by the off-by-one, and 41's 0/50 remains anomalous. Check Part 2b's")
    print("      stability numbers and consider that 41's specific pool draw was unusual.")

print()
print(BAR)
print("VERDICT 2 -- DOES p-IgGen's LAYER DIFFERENCE SURVIVE UNDER BOTH CONVENTIONS?")
print(BAR)
layer_rows = []
for conv in ("hook", "legacy"):
    e, l = summary[f"L{LAYER_EARLY}_{conv.upper()}"], summary[f"L{LAYER_LATE}_{conv.upper()}"]
    p_c, p_p, d_r, d_p = compare(e, l)
    ok_c, ok_p = holm2(p_c, p_p)
    sig = (ok_c and abs(d_r) >= MIN_RATE_SHIFT) or (ok_p and abs(d_p) >= MIN_PLDDT_SHIFT)
    layer_rows.append({"convention": conv, "early_rate": e["rate"], "late_rate": l["rate"],
                       "early_plddt": e["plddt_mean"], "late_plddt": l["plddt_mean"],
                       "diff_rate": d_r, "diff_plddt": d_p, "p_collapse": p_c, "p_plddt": p_p,
                       "holm_collapse": ok_c, "holm_plddt": ok_p, "layer_difference": bool(sig)})
    print(f"{conv:7s}: L{LAYER_EARLY} {e['rate']:.0%}/{e['plddt_mean']:.2f}  vs  "
          f"L{LAYER_LATE} {l['rate']:.0%}/{l['plddt_mean']:.2f}   "
          f"{d_r:+.1%} p={p_c:.4g}   {d_p:+.2f} p={p_p:.4g}   -> {'SIG' if sig else 'ns'}")

both = all(r["layer_difference"] for r in layer_rows)
either = any(r["layer_difference"] for r in layer_rows)
print()
if both:
    print("  ==> 1r-C IS ROBUST. The layer difference holds under both conventions. Report it.")
elif either:
    print("  ==> 1r-C IS CONVENTION-DEPENDENT. It appears under one extraction convention and not")
    print("      the other. That is a serious caveat and must be stated in the paper, not buried.")
else:
    print("  ==> The layer difference did NOT reproduce under either convention at N=100.")
    print("      1r-C does not survive replication -- retract it and say so plainly.")

print()
print("Sanity -- did each condition move vs CONTROL?")
for name, s in summary.items():
    if name == "CONTROL":
        continue
    p_c, p_p, d_r, d_p = compare(s, ctrl)
    print(f"  {name:14s} collapse {d_r:+.1%} p={p_c:.4g}   pLDDT {d_p:+.2f} p={p_p:.4g}")


VERDICT 1 -- DOES THE CONVENTION REPRODUCE THE 41-vs-57 SPLIT?
41 saw 0/50 collapse at layer 1 (alpha_rel 0.400); 57 saw 17/100 with the HOOK vector.

layer 1:  hook 9% / 71.33    legacy 9% / 70.55
          difference +0.0% collapse (p=1), +0.78 pLDDT (p=0.4027)   -> same
layer 3:  hook 4% / 73.01    legacy 1% / 74.19
          difference +3.0% collapse (p=0.3687), -1.18 pLDDT (p=0.3302)   -> same

  ==> The convention does NOT change the outcome here either. 1r-D is therefore NOT
      explained by the off-by-one, and 41's 0/50 remains anomalous. Check Part 2b's
      stability numbers and consider that 41's specific pool draw was unusual.

VERDICT 2 -- DOES p-IgGen's LAYER DIFFERENCE SURVIVE UNDER BOTH CONVENTIONS?
hook   : L1 9%/71.33  vs  L3 4%/73.01   +5.0% p=0.2507   -1.68 p=0.07271   -> ns
legacy : L1 9%/70.55  vs  L3 1%/74.19   +8.0% p=0.01849   -3.64 p=0.0004779   -> SIG

  ==> 1r-C IS CONVENTION-DEPENDENT. It appears under one extraction convention and not
      the other. T

In [12]:
# --- Persist. ---

rows = []
for name, recs in conditions.items():
    m = cond_meta[name]
    for i, r in enumerate(recs):
        rows.append({"model": "opig/p-IgGen", "condition": name, "idx": i, "layer": m["layer"],
                     "convention": m["convention"], "alpha_rel": DOSE * anchor_alpha_rel,
                     "target_len": TARGET_LEN, "sequence": r["sequence"],
                     "usable_length": sum(1 for a in r["sequence"] if a in VALID_AA),
                     "reached_target": r["reached_target"], "entropy": r["entropy"],
                     "plddt": r["plddt"], "ptm": r["ptm"], "fold_ok": r["fold_ok"],
                     "collapse": r["collapse"]})
pd.DataFrame(rows).to_csv("piggen_convention_sequences.csv", index=False)

pd.DataFrame([{"condition": n, "collapsed": s["k"], "n": s["n"], "collapse_rate": s["rate"],
               "ci_lo": s["ci"][0], "ci_hi": s["ci"][1], "fold_ok": s["fold_ok"],
               "mean_plddt": s["plddt_mean"], "mean_entropy": s["entropy"],
               "mean_len": s["mean_len"]} for n, s in summary.items()]
             ).to_csv("piggen_convention_summary.csv", index=False)

pd.DataFrame(layer_rows).to_csv("piggen_convention_layer_tests.csv", index=False)
pd.DataFrame(conv_rows).to_csv("piggen_convention_contrasts.csv", index=False)

cos_rows = []
for L in (LAYER_EARLY, LAYER_LATE):
    cos_rows.append({"layer": L, "metric": "convention", "value": conv_cos[L],
                     "sd": float("nan"), "min": float("nan")})
for (conv, L), (m, sd, mn) in stab.items():
    cos_rows.append({"layer": L, "metric": f"split_half_{conv}", "value": m, "sd": sd, "min": mn})
pd.DataFrame(cos_rows).to_csv("piggen_vector_cosines.csv", index=False)

pd.DataFrame([{
    "model": "opig/p-IgGen", "n_layers": N_LAYERS, "anchor_alpha_rel": anchor_alpha_rel,
    "dose_mult": DOSE, "alpha_rel": DOSE * anchor_alpha_rel, "target_len": TARGET_LEN,
    "n_per_arm": N_PER_CONDITION,
    "h_early": h_model[LAYER_EARLY], "h_late": h_model[LAYER_LATE],
    "cos_convention_early": conv_cos[LAYER_EARLY], "cos_convention_late": conv_cos[LAYER_LATE],
    "cos_splithalf_hook_early": stab[("hook", LAYER_EARLY)][0],
    "cos_splithalf_hook_late": stab[("hook", LAYER_LATE)][0],
    "cos_splithalf_legacy_early": stab[("legacy", LAYER_EARLY)][0],
    "cos_splithalf_legacy_late": stab[("legacy", LAYER_LATE)][0],
    "convention_changes_outcome": bool(conv_matters),
    "layer_diff_both_conventions": bool(both),
    "pool_collapse_rate": pool_collapse, "pool_mean_plddt": pool_plddt,
    "utility_gap": util_gap, "repetition_separation": rep_gap,
    "used_uniprot_fallback": USED_FALLBACK, "supports_min_new_tokens": SUPPORTS_MIN_NEW,
}]).to_csv("piggen_convention_calibration.csv", index=False)

print("Saved:")
for f in ["piggen_convention_sequences.csv", "piggen_convention_summary.csv",
          "piggen_convention_layer_tests.csv", "piggen_convention_contrasts.csv",
          "piggen_vector_cosines.csv",
          "piggen_convention_calibration.csv"]:
    print("  " + f)
print()
print("DOWNLOAD THE OUTPUT TAB BEFORE CLOSING THE SESSION.")
print()
print("piggen_vector_cosines.csv is the one to keep regardless of how Part 3 lands -- the")
print("split-half numbers are the first measurement of steering-vector reproducibility anywhere")
print("in this project, and they apply to every model, not just this one.")


Saved:
  piggen_convention_sequences.csv
  piggen_convention_summary.csv
  piggen_convention_layer_tests.csv
  piggen_convention_contrasts.csv
  piggen_vector_cosines.csv
  piggen_convention_calibration.csv

DOWNLOAD THE OUTPUT TAB BEFORE CLOSING THE SESSION.

piggen_vector_cosines.csv is the one to keep regardless of how Part 3 lands -- the
split-half numbers are the first measurement of steering-vector reproducibility anywhere
in this project, and they apply to every model, not just this one.
